# CarRacingWObstacles: PPO in Google Colab

    This notebook installs the environment, trains PPO for **500 completed training episodes**,
    plots total reward per episode (raw and trailing mean over 100 episodes), and displays
    an MP4 rollout of the **best validation checkpoint**. Run the cells in order.

    In Colab choose **Runtime → Change runtime type → T4 GPU** if available. The example uses four RGB frames and holds each action for three physics frames.

    Validation runs every 25 training episodes on five reserved validation seeds.
    These additional validation episodes do not count toward the 500 training episodes.
    The ten final test cases are used only after checkpoint selection. A successful policy
    is an experimental outcome; 500 episodes is the training budget, not a success guarantee.

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"

REPO = Path("/content/CarRacingWObstacles")
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/luisgarrote/CarRacingWObstacles.git", str(REPO)])
# Install SWIG first: Box2D may need to build its Python bindings.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade",
                       "pip", "setuptools", "wheel", "swig"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-build-isolation",
                       "-e", str(REPO) + "[sb3]"])

In [ ]:
import csv, json, math, time, sys
import importlib.metadata
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Video
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

EXAMPLES = REPO / "examples"
sys.path.insert(0, str(EXAMPLES))
from env_utils import make_env, prepare_env, episode, get_case, load_cases, save_results
from colab_callbacks import ValidateAndStop

torch.set_num_threads(1)

## Experiment settings

    Change the seed and output directory for each independent run. Keep validation
    and final test seeds out of training. This demo uses one environment so the
    episode callback counts exactly 500 episodes across the run.

In [ ]:
TRAIN_EPISODES = 500
SEED = 0
HISTORY = 4
ACTION_REPEAT = 3
MAX_STEPS = 1000  # physics frames per episode
ROLLOUT = 512
BATCH_SIZE = 128
EPOCHS = 4
VALIDATE_EVERY = 25
VALIDATION_SEEDS = [10001, 10002, 10003, 10004, 10005]
RANDOMIZE = False
TRAFFIC = 0  # set 1 or 2 to train with moving obstacles
OBSTACLES = 0
POLICY = "CnnPolicy"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN = REPO / "outputs" / ("colab_ppo_seed" + str(SEED) + "_" + str(time.time_ns()))
RUN.mkdir(parents=True)
print("Device:", DEVICE, "| Output:", RUN)

In [ ]:
def raw_env():
    return make_env(max_steps=MAX_STEPS, randomize=RANDOMIZE,
                    traffic=TRAFFIC, obstacles=OBSTACLES, action_repeat=ACTION_REPEAT)

train_env = prepare_env(Monitor(raw_env(), filename=str(RUN / "training")), HISTORY)
validation_env = prepare_env(raw_env(), HISTORY)
train_env.seed(SEED)
print("Observation:", train_env.observation_space)
print("Action:", train_env.action_space)

## Train PPO and select the best checkpoint

    The PPO calls below are ordinary SB3. `ValidateAndStop` evaluates deterministic
    actions on the same validation seeds, saves `best_model.zip` whenever mean raw
    validation return improves, and stops at 500 completed training episodes.
    See `examples/colab_callbacks.py` for its short implementation.

    `model.zip` stores the final policy; the video below explicitly loads
    `best_model.zip`. Training and validation rewards are never normalized.

In [ ]:
callback = ValidateAndStop(RUN, validation_env, VALIDATION_SEEDS,
                           max_episodes=TRAIN_EPISODES, every=VALIDATE_EVERY)
# SB3 budgets count policy decisions. The callback enforces the episode limit.
decision_budget = TRAIN_EPISODES * math.ceil(MAX_STEPS / ACTION_REPEAT) + ROLLOUT
config = dict(algorithm="PPO", policy=POLICY, history=HISTORY, action_repeat=ACTION_REPEAT,
              seed=SEED, episodes=TRAIN_EPISODES, max_steps=MAX_STEPS, device=DEVICE,
              randomize=RANDOMIZE, traffic=TRAFFIC, obstacles=OBSTACLES,
              validation_seeds=VALIDATION_SEEDS, validate_every=VALIDATE_EVERY,
              versions={name: importlib.metadata.version(name) for name in
                        ("stable-baselines3", "torch", "gymnasium", "numpy")})
(RUN / "config.json").write_text(json.dumps(config, indent=2))
try:
    model = PPO(POLICY, train_env, seed=SEED, device=DEVICE, verbose=0,
                n_steps=ROLLOUT, batch_size=BATCH_SIZE, n_epochs=EPOCHS,
                learning_rate=3e-4)
    model.learn(total_timesteps=decision_budget, callback=callback)
    model.save(str(RUN / "model"))
    config.update(completed_episodes=len(callback.training_returns),
                  actual_timesteps=model.num_timesteps, simulator_frames=callback.simulator_frames)
    (RUN / "config.json").write_text(json.dumps(config, indent=2))
finally:
    train_env.close()
    validation_env.close()
assert config["completed_episodes"] == TRAIN_EPISODES
print("Best checkpoint:", json.loads((RUN / "best_model.json").read_text()))

## Total reward per completed training episode

    The thin curve shows each episode's raw reward sum. The thick curve is the mean
    of the most recent 100 episodes; before episode 100 it uses the available episodes.
    The validation episodes are not included in this training graph.

In [ ]:
with (RUN / "training.monitor.csv").open() as stream:
    rows = list(csv.DictReader(line for line in stream if not line.startswith("#")))
rewards = np.array([float(row["r"]) for row in rows])
episodes = np.arange(1, len(rewards) + 1)
average = np.array([rewards[max(0, i - 99):i + 1].mean() for i in range(len(rewards))])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, rewards, alpha=0.4, linewidth=1, label="Raw episode return")
ax.plot(episodes, average, linewidth=2, label="Mean of last 100 episodes")
ax.set(xlabel="Completed training episode", ylabel="Total reward")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(RUN / "rewards.png", dpi=160)
display(fig)
plt.close(fig)
np.savetxt(RUN / "rewards.csv", np.column_stack([episodes, rewards, average]),
           delimiter=",", header="episode,raw_return,mean_last_100", comments="")

## Watch the best model

    This is a deterministic rollout of `best_model.zip` on `normal_1`, a final test
    case that was excluded from checkpoint selection. The MP4 captures every physics
    frame, including intermediate frames during action persistence and the terminal
    frame. The recorder streams frames to disk rather than storing them all in memory.

In [ ]:
import imageio.v2 as imageio
from record_video import VideoFrames
from env_utils import VIDEO_FPS
from action_repeat import ActionRepeat
demo_case = get_case("normal_1")
video_path = RUN / "best_model_demo.mp4"
with imageio.get_writer(str(video_path), fps=VIDEO_FPS, codec="libx264", macro_block_size=1) as writer:
    base = make_env(demo_case)
    recorded = VideoFrames(base, writer)
    recorded = ActionRepeat(recorded, ACTION_REPEAT)
    video_env = prepare_env(recorded, HISTORY)
    try:
        best = PPO.load(str(RUN / "best_model.zip"), env=video_env, device=DEVICE)
        video_result = episode(best, video_env, demo_case["seed"])
    finally:
        video_env.close()
(RUN / "video_result.json").write_text(json.dumps(dict(case=demo_case["id"], **video_result), indent=2))
plt.close("all")
print(video_result)
display(Video(str(video_path), embed=True, width=640))

## Final evaluation: five + five fixed cases

    Evaluate the selected checkpoint once on all ten supplied cases. Keep failed
    episodes. The CSV and JSON contain every return and each group's mean, standard
    deviation and success rate. These scores do not select another checkpoint.

In [ ]:
best = PPO.load(str(RUN / "best_model.zip"), device=DEVICE)
results = []
for case in load_cases():
    base = make_env(case, action_repeat=ACTION_REPEAT)
    test_env = prepare_env(base, HISTORY)
    try:
        result = episode(best, test_env, case["seed"])
        results.append(dict(case_id=case["id"], group=case["group"], **result))
    finally:
        test_env.close()
summary = save_results(results, RUN / "best_model_evaluation")

## Download the results (optional)

    The archive includes the final and best models, model-selection metadata,
    configuration, training/validation logs, reward graph, test statistics and video.

In [ ]:
def download_run():
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
    files.download(archive)

# Uncomment to download all run artifacts:
# download_run()